In [2]:
# 08_history_csv.py — build the 5-day History CSV for the dashboard from the trusted eval.
# Reuses 06_eval_refit2023_fx functions. One lead-matched pass, resumable.
# Output: reports/tft_zonal_predictions.csv  (date, actual_day1..5, pred_day1..5 = daily total MW)
import importlib.util, os
import numpy as np, pandas as pd, torch

ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
EVAL = os.path.join(ROOT, "06_eval_refit2023_fx.py")
OUT  = os.path.join(ROOT, "..", "reports", "tft_zonal_predictions.csv")

spec = importlib.util.spec_from_file_location("evalmod", EVAL)
ev = importlib.util.module_from_spec(spec); spec.loader.exec_module(ev)

def cache(d): return os.path.join(ROOT, f"hist_lead{d}.npz")

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    df = ev.load_base()
    leads = pd.read_parquet(ev.LEADS); leads.index = pd.to_datetime(leads.index)
    if getattr(leads.index, "tz", None) is not None:
        leads.index = leads.index.tz_localize(None)

    one = df[df["zone"] == df["zone"].iloc[0]][["time_idx", "utc"]]
    idx2utc = dict(zip(one["time_idx"], one["utc"]))

    tsi = int(df.loc[df["utc"] >= ev.TEST_START, "time_idx"].min())
    training_ds = ev.build_training_ds(df)
    model = ev.TemporalFusionTransformer.load_from_checkpoint(ev.CKPT).to(device).eval()

    df = df[df["time_idx"] >= tsi - ev.ENCODER_LEN - 1].copy()
    ZONES = sorted(df["zone"].unique())
    print(f"windows from idx {tsi}, device={device}", flush=True)

    # per lead d, daily total for THAT horizon's day, keyed by decoder-start idx
    pred_by_lead, act_by_lead = {}, {}
    for d in range(1, 6):
        print(f"\n=== Lead {d} ===", flush=True)
        if os.path.exists(cache(d)):
            z = np.load(cache(d)); ut, ph, ah = z["ut"], z["pred"], z["act"]
            print("  cached", flush=True)
        else:
            dfx = ev.swap_lead(df, leads, d)
            hat, true = {}, {}
            for zi, zone in enumerate(ZONES):
                yh, yt, tt = ev.predict_one_zone(model, training_ds,
                                                 dfx[dfx["zone"] == zone], tsi, device)
                for i, t in enumerate(tt):
                    hat.setdefault(int(t), []).append(yh[i])
                    true.setdefault(int(t), []).append(yt[i])
                print(f"    {zone:10s} ({zi+1}/11)", flush=True)
            del dfx
            good = sorted(t for t, v in hat.items() if len(v) == 11)
            s, e = (d - 1) * 24, d * 24              # this horizon's 24h slice
            ut = np.array(good)
            ph = np.array([np.sum(hat[t], axis=0)[s:e].sum() for t in good])   # daily total MW
            ah = np.array([np.sum(true[t], axis=0)[s:e].sum() for t in good])
            np.savez(cache(d), ut=ut, pred=ph, act=ah)
            del hat, true
        pred_by_lead[d] = dict(zip(ut.tolist(), ph.tolist()))
        act_by_lead[d]  = dict(zip(ut.tolist(), ah.tolist()))

    # keep only windows whose decoder starts at local midnight -> one row per date
    rows = []
    for t, utc in idx2utc.items():
        local = pd.Timestamp(utc, tz="UTC").tz_convert("America/New_York")
        if local.hour != 0:
            continue
        if not all(t in pred_by_lead[d] for d in range(1, 6)):
            continue
        row = {"date": local.date()}
        for d in range(1, 6):
            row[f"pred_day{d}"] = round(pred_by_lead[d][t], 1)
            row[f"actual_day{d}"] = round(act_by_lead[d][t], 1)
        rows.append(row)

    out = pd.DataFrame(rows).sort_values("date")
    cols = ["date"] + [f"{p}_day{d}" for d in range(1, 6) for p in ("actual", "pred")]
    out = out[cols]
    os.makedirs(os.path.dirname(OUT), exist_ok=True)
    out.to_csv(OUT, index=False)
    print(f"\nwrote {len(out)} dates -> {os.path.abspath(OUT)}", flush=True)

    # sanity: MAPE per horizon from the daily totals should track your eval
    for d in range(1, 6):
        a, p = out[f"actual_day{d}"], out[f"pred_day{d}"]
        m = (np.abs(a - p) / a).mean() * 100
        print(f"  day{d} daily-total MAPE: {m:.2f}%", flush=True)

if __name__ == "__main__":
    main()

windows from idx 75080, device=cuda

=== Lead 1 ===
  cached

=== Lead 2 ===
  cached

=== Lead 3 ===
  cached

=== Lead 4 ===
  cached

=== Lead 5 ===


/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


  cached

wrote 854 dates -> /opt/app-root/src/Forecasting-Energy-Demand/reports/tft_zonal_predictions.csv
  day1 daily-total MAPE: 1.43%
  day2 daily-total MAPE: 1.84%
  day3 daily-total MAPE: 2.11%
  day4 daily-total MAPE: 2.58%
  day5 daily-total MAPE: 2.95%


In [3]:
ls -l /opt/app-root/src/Forecasting-Energy-Demand/reports/tft_zonal_predictions.csv
head -3 /opt/app-root/src/Forecasting-Energy-Demand/reports/tft_zonal_predictions.csv

NameError: name 'ls' is not defined